## Multi-container pods — the sidecar pattern

The pod earns its keep when you add a *helper* container. The **sidecar pattern**: a small companion that adds a capability to the main container **without changing its image**. Common examples — a log shipper tailing the app's files, a service-mesh proxy (Envoy, `linkerd-proxy`) intercepting its traffic, a config reloader watching a mount.

What makes it work is the **shared volume** and **shared network**.

```yaml
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh","-c","while true; do date >> /var/log/app.log; sleep 2; done"]
      volumeMounts: [{ name: logs, mountPath: /var/log }]
    - name: log-shipper
      image: busybox:1.36
      command: ["sh","-c","tail -F /var/log/app.log"]
      volumeMounts: [{ name: logs, mountPath: /var/log }]
  volumes:
    - name: logs
      emptyDir: {}
```

`app` writes to `/var/log/app.log`; `log-shipper` `tail -F`s the same file — different container, same file, because the volume is mounted in both. Read either with `kubectl logs <pod> -c <container>`; the **`-c` flag is required** when a pod has more than one container.

**Shared network — proof on `localhost`.** If the main container listens on `:8080`, the sidecar reaches it at `localhost:8080` — same network namespace, no DNS. That's how an Envoy sidecar transparently intercepts traffic.

**When *not* to.** Two independent services that call each other over the network belong in **separate pods** behind a Service. Co-locate only when they must share a node, share a lifecycle, and share network/storage. The classic mistake — a web app and its database in one pod — bundles two things with independent scaling and lifecycle. On our map this is the **sidecar** chip in the **Pod** box: a companion, not a second service.